In [0]:
bronze_shrinkage_type = spark.table(
    "workforce_analytics.bronze.dim_shrinkage_type_raw"
)

bronze_shrinkage_type.printSchema()

display(bronze_shrinkage_type)

Step 1: Inspect the original values

In [0]:
display(
    bronze_shrinkage_type
    .select("is_paid", "is_planned")
    .distinct()
)

Step 2: Create the cleaned table

In [0]:
from pyspark.sql import functions as F

silver_shrinkage_type = bronze_shrinkage_type.select(
    F.trim(F.col("shrinkage_type_id")).alias("shrinkage_type_id"),
    F.trim(F.col("shrinkage_type")).alias("shrinkage_type"),
    F.trim(F.col("category")).alias("category"),
    F.lower(F.trim(F.col("is_paid"))).cast("boolean").alias("is_paid"),
    F.lower(F.trim(F.col("is_planned"))).cast("boolean").alias("is_planned"),
    F.col("_source_file"),
    F.col("_ingested_at")
)

Step 3: Validate the data

In [0]:
# Check schema
silver_shrinkage_type.printSchema()

# View records
display(silver_shrinkage_type)

# Check total rows
print("Total rows:", silver_shrinkage_type.count())

# Check missing IDs
print(
    "Missing shrinkage_type_id:",
    silver_shrinkage_type.filter(
        F.col("shrinkage_type_id").isNull() |
        (F.col("shrinkage_type_id") == "")
    ).count()
)

# Check duplicate IDs
duplicate_shrinkage_types = (
    silver_shrinkage_type
    .groupBy("shrinkage_type_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate IDs:", duplicate_shrinkage_types.count())

display(duplicate_shrinkage_types)

In [0]:
silver_shrinkage_type.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workforce_analytics.silver.dim_shrinkage_type")

saved_shrinkage_type = spark.table(
    "workforce_analytics.silver.dim_shrinkage_type"
)

print("Silver DimShrinkageType rows:", saved_shrinkage_type.count())

display(saved_shrinkage_type)